# Cross-city weather → expected-energy feature engineering

This notebook creates a broad, **leakage-safe candidate feature set** for Las Vegas and Phoenix. It is designed for subsequent feature selection in a training notebook, including a weather-transfer model and an operational model with historical-load features.

**Important:** rows retain missing values produced by lags and rolling windows. Do not fill, scale, select, or target-encode features here; fit those operations on each training split only.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

BASE = Path.cwd()
INPUTS = {
    'las_vegas': BASE / 'processed_lstm_baseline_deviation' / 'las_vegas' / 'las_vegas_featured.csv',
    'phoenix': BASE / 'processed_lstm_baseline_deviation' / 'phoenix' / 'phoenix_featured.csv',
}
OUTPUT_DIR = BASE / 'processed_feature_candidates_cross_city_v1'
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = 'hourly_load_mw'
LAGS = (1, 3, 6, 12, 24, 48, 72)
TARGET_LAGS = (1, 2, 3, 6, 12, 24, 48, 72, 168)
ROLL_WINDOWS = (3, 6, 12, 24, 72)
print('Writing feature candidates to:', OUTPUT_DIR)

Writing feature candidates to: D:\Downloads\BTP\model_training\lstm_training_3\processed_feature_candidates_cross_city_v1


## Feature policy

- **Transfer core**: calendar, contemporaneous/forecast weather, deterministic solar proxies, and historical weather features.
- **Operational-only**: historical-load lags and trailing summaries. They are valid only when past metered load is known at prediction time; exclude them from cold-start cross-city evaluation.
- **Experimental/local**: visibility, cloud, aerosol, snow, and weather-code-related features. Keep them for later ablation, not as a presumed transferable core.
- A 25-hour gap starts a new segment. No lag or rolling feature crosses a gap.

In [2]:
def add_feature(registry, frame, name, values, group, policy, description):
    frame[name] = values
    registry.append({'feature': name, 'group': group, 'policy': policy, 'description': description})

def segmented_shift(frame, col, periods):
    return frame.groupby('_segment_id', sort=False)[col].shift(periods)

def segmented_rolling(frame, col, window, stat):
    # shift first: the row at timestamp t never contributes to its own feature
    prior = frame.groupby('_segment_id', sort=False)[col].shift(1)
    grouped = prior.groupby(frame['_segment_id'], sort=False)
    if stat == 'mean':
        return grouped.transform(lambda s: s.rolling(window, min_periods=window).mean())
    if stat == 'max':
        return grouped.transform(lambda s: s.rolling(window, min_periods=window).max())
    if stat == 'min':
        return grouped.transform(lambda s: s.rolling(window, min_periods=window).min())
    if stat == 'std':
        return grouped.transform(lambda s: s.rolling(window, min_periods=window).std())
    raise ValueError(stat)

def build_features(city, path):
    df = pd.read_csv(path, parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    df.insert(1, 'city', city)
    gap = df['timestamp'].diff().ne(pd.Timedelta(hours=1))
    gap.iloc[0] = True
    df['_segment_id'] = gap.cumsum().astype('int16')
    registry = []

    # Preserve original columns as candidates and document their provenance.
    for col in df.columns:
        if col not in {'timestamp', 'city', TARGET, '_segment_id'}:
            registry.append({'feature': col, 'group': 'existing_raw_or_engineered', 'policy': 'candidate',
                             'description': 'Present in the existing merged feature table.'})

    ts = df['timestamp']
    hour = ts.dt.hour
    dow = ts.dt.dayofweek
    doy = ts.dt.dayofyear
    add_feature(registry, df, 'year_candidate', ts.dt.year, 'calendar_optional', 'candidate', 'Year: useful for drift analysis; normally exclude for cross-city transfer.')
    add_feature(registry, df, 'is_holiday_us_federal', ts.dt.normalize().isin(USFederalHolidayCalendar().holidays(ts.min(), ts.max())).astype('int8'), 'calendar', 'transfer_core', 'US federal holiday flag.')
    add_feature(registry, df, 'is_business_hour', ((hour >= 8) & (hour < 18) & (dow < 5)).astype('int8'), 'calendar', 'transfer_core', 'Working-hour proxy.')
    add_feature(registry, df, 'is_night', ((hour < 6) | (hour >= 22)).astype('int8'), 'calendar', 'transfer_core', 'Night-time proxy.')
    add_feature(registry, df, 'week_of_year', ts.dt.isocalendar().week.astype('int16'), 'calendar_optional', 'candidate', 'Week-of-year; test against cyclic annual encoding.')

    # Weather availability: only make features for columns that actually exist.
    temp = df['temperature_c']
    dew = df['dew_point_c']
    rh = df['relative_humidity_percent']
    wind = df['wind_speed_ms']
    direction = np.deg2rad(df['wind_direction_deg'])
    add_feature(registry, df, 'wind_direction_sin', np.sin(direction), 'wind', 'transfer_core', 'Circular wind-direction encoding.')
    add_feature(registry, df, 'wind_direction_cos', np.cos(direction), 'wind', 'transfer_core', 'Circular wind-direction encoding.')
    add_feature(registry, df, 'wind_u_component', -wind * np.sin(direction), 'wind', 'candidate', 'East-west wind component.')
    add_feature(registry, df, 'wind_v_component', -wind * np.cos(direction), 'wind', 'candidate', 'North-south wind component.')
    add_feature(registry, df, 'temperature_dewpoint_spread_c', temp - dew, 'thermal_moisture', 'transfer_core', 'Dryness / dew-point depression.')
    add_feature(registry, df, 'temperature_humidity_interaction_v2', temp * rh, 'thermal_moisture', 'candidate', 'Temperature × relative humidity.')
    add_feature(registry, df, 'temperature_dewpoint_interaction', temp * dew, 'thermal_moisture', 'candidate', 'Temperature × dew point.')
    vapor_pressure = 6.11 * np.exp(5417.7530 * (1 / 273.16 - 1 / (273.15 + dew)))
    add_feature(registry, df, 'humidex_c', temp + (5 / 9) * (vapor_pressure - 10), 'thermal_moisture', 'candidate', 'Humidity-adjusted apparent temperature (humidex).')
    for base in (15, 18, 20, 22, 24):
        add_feature(registry, df, f'cooling_degree_base_{base}c', (temp - base).clip(lower=0), 'thermal_nonlinear', 'transfer_core', f'Cooling degree hours at {base}°C.')
        add_feature(registry, df, f'heating_degree_base_{base}c', (base - temp).clip(lower=0), 'thermal_nonlinear', 'transfer_core', f'Heating degree hours at {base}°C.')
    add_feature(registry, df, 'temperature_cubed', temp ** 3, 'thermal_nonlinear', 'candidate', 'Cubic temperature term for linear models only.')
    for col in ('hourly_precipitation_cm', 'ceiling_height_m', 'horizontal_visibility_km', 'broadband_aerosol_optical_depth', 'precipitable_water_mm'):
        if col in df:
            add_feature(registry, df, f'log1p_{col}', np.log1p(df[col].clip(lower=0)), 'transforms', 'candidate', f'Log transform of {col}.')
    if 'hourly_precipitation_cm' in df:
        add_feature(registry, df, 'is_precipitating', (df['hourly_precipitation_cm'] > 0).astype('int8'), 'weather_event', 'candidate', 'Nonzero precipitation event.')
    if 'total_sky_cover_tenths' in df and 'opaque_sky_cover_tenths' in df:
        add_feature(registry, df, 'cloud_opaque_fraction', df['opaque_sky_cover_tenths'] / 10, 'cloud_local', 'candidate', 'Opaque sky fraction.')
        add_feature(registry, df, 'cloud_nonopaque_tenths', df['total_sky_cover_tenths'] - df['opaque_sky_cover_tenths'], 'cloud_local', 'candidate', 'Total minus opaque cloud cover.')

    # Explicit calendar × weather interactions, justified by the prior hour/season diagnostics.
    for name, base_values in {'temperature': temp, 'dewpoint': dew, 'humidex': df['humidex_c']}.items():
        add_feature(registry, df, f'{name}_x_sin_hour', base_values * np.sin(2 * np.pi * hour / 24), 'time_weather_interaction', 'transfer_core', f'{name} × cyclic hour sine.')
        add_feature(registry, df, f'{name}_x_cos_hour', base_values * np.cos(2 * np.pi * hour / 24), 'time_weather_interaction', 'transfer_core', f'{name} × cyclic hour cosine.')
        add_feature(registry, df, f'{name}_x_weekend', base_values * (dow >= 5), 'time_weather_interaction', 'candidate', f'{name} × weekend.')
    for base in (18, 22):
        cooling = df[f'cooling_degree_base_{base}c']
        add_feature(registry, df, f'cooling_degree_{base}c_x_sin_hour', cooling * np.sin(2 * np.pi * hour / 24), 'time_weather_interaction', 'candidate', f'Cooling degree {base}°C × hour sine.')
        add_feature(registry, df, f'cooling_degree_{base}c_x_cos_hour', cooling * np.cos(2 * np.pi * hour / 24), 'time_weather_interaction', 'candidate', f'Cooling degree {base}°C × hour cosine.')

    lag_weather = [c for c in ('temperature_c', 'dew_point_c', 'relative_humidity_percent', 'wind_speed_ms', 'station_pressure_kpa', 'precipitable_water_mm', 'hourly_precipitation_cm', 'total_sky_cover_tenths', 'opaque_sky_cover_tenths') if c in df]
    for col in lag_weather:
        for lag in LAGS:
            add_feature(registry, df, f'{col}_lag_{lag}h', segmented_shift(df, col, lag), 'weather_lag', 'transfer_core', f'{col} observed {lag} hours earlier; never crosses a time gap.')
        for window in ROLL_WINDOWS:
            add_feature(registry, df, f'{col}_prior_{window}h_mean', segmented_rolling(df, col, window, 'mean'), 'weather_rolling', 'transfer_core', f'Prior {window}-hour mean of {col}.')
        if col in {'temperature_c', 'dew_point_c', 'wind_speed_ms', 'hourly_precipitation_cm'}:
            for window in (6, 24):
                add_feature(registry, df, f'{col}_prior_{window}h_max', segmented_rolling(df, col, window, 'max'), 'weather_rolling', 'candidate', f'Prior {window}-hour maximum of {col}.')
                add_feature(registry, df, f'{col}_prior_{window}h_min', segmented_rolling(df, col, window, 'min'), 'weather_rolling', 'candidate', f'Prior {window}-hour minimum of {col}.')
    for lag in (1, 3, 6, 24):
        add_feature(registry, df, f'temperature_change_{lag}h', temp - segmented_shift(df, 'temperature_c', lag), 'weather_change', 'transfer_core', f'Current temperature minus temperature {lag} hours earlier.')
        add_feature(registry, df, f'dewpoint_change_{lag}h', dew - segmented_shift(df, 'dew_point_c', lag), 'weather_change', 'candidate', f'Current dew point minus dew point {lag} hours earlier.')

    # Historical load is intentionally separate: valid operationally, excluded for cold-start transfer.
    add_feature(registry, df, 'target_log1p', np.log1p(df[TARGET]), 'target', 'target_only', 'Optional target transform; do not include as an input.')
    for lag in TARGET_LAGS:
        add_feature(registry, df, f'load_lag_{lag}h', segmented_shift(df, TARGET, lag), 'load_history', 'operational_only', f'Load observed {lag} hours earlier; exclude from zero-history cross-city tests.')
    for window in (3, 6, 24, 72, 168):
        add_feature(registry, df, f'load_prior_{window}h_mean', segmented_rolling(df, TARGET, window, 'mean'), 'load_history', 'operational_only', f'Prior {window}-hour load mean.')
    for window in (24, 168):
        add_feature(registry, df, f'load_prior_{window}h_std', segmented_rolling(df, TARGET, window, 'std'), 'load_history', 'operational_only', f'Prior {window}-hour load variability.')

    # Keep technical columns for audit but remove the segmentation key from modelling candidates.
    manifest = pd.DataFrame(registry).drop_duplicates('feature', keep='last').sort_values(['policy', 'group', 'feature'])
    return df, manifest


In [3]:
all_manifests = []
summary = []
for city, path in INPUTS.items():
    city_df, city_manifest = build_features(city, path)
    # CSV only: portable across the existing GPU environment without optional engines.
    output = OUTPUT_DIR / f'{city}_feature_candidates.csv'
    city_df.to_csv(output, index=False)
    storage_format = 'csv'
    city_manifest['city'] = city
    all_manifests.append(city_manifest)
    summary.append({'city': city, 'rows': len(city_df), 'columns_including_targets_and_audit': len(city_df.columns), 'format': storage_format, 'output': str(output)})
    print(city, city_df.shape, '->', output.name)

manifest = pd.concat(all_manifests, ignore_index=True).drop_duplicates(['feature', 'policy', 'group']).sort_values(['policy', 'group', 'feature'])
manifest.to_csv(OUTPUT_DIR / 'feature_manifest.csv', index=False)
pd.DataFrame(summary).to_csv(OUTPUT_DIR / 'generation_summary.csv', index=False)
display(pd.DataFrame(summary))
display(manifest.groupby(['policy', 'group']).size().rename('feature_count').reset_index())

C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = values
C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = values
C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

las_vegas (70080, 233) -> las_vegas_feature_candidates.csv


C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = values
C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = values
C:\Users\HP VICTUS\AppData\Local\Temp\ipykernel_23132\650949874.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

phoenix (70080, 233) -> phoenix_feature_candidates.csv


,city,rows,columns_including_targets_and_audit,format,output
0,las_vegas,70080,233,csv,D:\Downloads\BTP\model_training\lstm_training_...
1,phoenix,70080,233,csv,D:\Downloads\BTP\model_training\lstm_training_...


,policy,group,feature_count
0,candidate,calendar_optional,2
1,candidate,cloud_local,2
2,candidate,existing_raw_or_engineered,35
3,candidate,thermal_moisture,3
4,candidate,thermal_nonlinear,1
5,candidate,time_weather_interaction,7
6,candidate,transforms,5
7,candidate,weather_change,4
8,candidate,weather_event,1
9,candidate,weather_rolling,16


## How to use this output in the training notebook

1. Read a city parquet file and filter the manifest by `policy`.
2. For cross-city transfer, start with `transfer_core`; exclude `load_history`, `target`, `year_candidate`, and local-only weather groups unless an ablation proves they help on held-out cities.
3. Split by time/city first. Fit imputation, scaling, spline thresholds, PCA, target encoding, and feature selection only on the training partition.
4. Compare calendar-only, transfer-core, transfer-core + candidate groups, and operational-only load-history additions.
5. For LSTM/GRU, use raw sequential weather/calendar features; do not feed every hand-built lag as well unless an ablation shows value.